In [1]:
"""
Global Superstore - Phase 2 | Notebook 05
Sales Forecasting with Prophet
File: 02_Python/05_Forecast.py

Chay: python 02_Python/05_Forecast.py
Output: charts/chart_19_20.png + data/forecast_result.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────────
DB_FILE   = r"C:\DA\global-superstore-analysis\data\superstore.db"
CHART_DIR = r"C:\DA\global-superstore-analysis\02_Python\charts"
DATA_DIR  = r"C:\DA\global-superstore-analysis\data"
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

BLUE  = "#2E75B6"
RED   = "#C0392B"
GREEN = "#27AE60"
GRAY  = "#BDC3C7"

print("=" * 60)
print("GLOBAL SUPERSTORE - SALES FORECASTING")
print("=" * 60)

# ── Load data ──────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_FILE)
df   = pd.read_sql("SELECT * FROM orders", conn)
conn.close()

df["order_date"] = pd.to_datetime(df["order_date"])
print(f"\nData loaded: {df.shape[0]:,} rows")
print(f"Date range : {df['order_date'].min().date()} -> {df['order_date'].max().date()}")

# ── Tong hop doanh thu theo thang ─────────────────────────────────────────────
monthly = df.groupby(df["order_date"].dt.to_period("M")).agg(
    sales  = ("sales",  "sum"),
    profit = ("profit", "sum"),
    orders = ("order_id","nunique"),
).reset_index()
monthly["order_date"] = monthly["order_date"].dt.to_timestamp()
monthly = monthly.sort_values("order_date").reset_index(drop=True)

print(f"\nMonthly data points: {len(monthly)}")
print(f"Avg monthly sales  : ${monthly['sales'].mean():,.0f}")
print(f"Max monthly sales  : ${monthly['sales'].max():,.0f}")

# ── Thu Prophet ───────────────────────────────────────────────────────────────
USE_PROPHET = False
try:
    from prophet import Prophet
    USE_PROPHET = True
    print("\nProphet found - using Prophet for forecasting")
except ImportError:
    print("\nProphet not found - using manual trend + seasonality model")

if USE_PROPHET:
    # ── PROPHET FORECAST ──────────────────────────────────────────────────────
    prophet_df = monthly[["order_date", "sales"]].rename(
        columns={"order_date": "ds", "sales": "y"}
    )

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=0.1,
    )
    model.fit(prophet_df)

    future    = model.make_future_dataframe(periods=6, freq="MS")
    forecast  = model.predict(future)

    forecast_future = forecast[forecast["ds"] > monthly["order_date"].max()].copy()
    forecast_hist   = forecast[forecast["ds"] <= monthly["order_date"].max()].copy()

    print(f"\nForecast next 6 months:")
    print(forecast_future[["ds","yhat","yhat_lower","yhat_upper"]].to_string(index=False))

else:
    # ── MANUAL TREND + SEASONALITY MODEL (khong can Prophet) ─────────────────
    print("\nBuilding manual forecast model...")

    # Tao features
    monthly["month_num"] = np.arange(len(monthly))
    monthly["month"]     = monthly["order_date"].dt.month
    monthly["year"]      = monthly["order_date"].dt.year

    # Trend (linear regression)
    x = monthly["month_num"].values
    y = monthly["sales"].values
    coeffs = np.polyfit(x, y, deg=2)   # bac 2 de bat xu huong tang
    trend_fn = np.poly1d(coeffs)

    # Seasonal factor theo thang (trung binh)
    monthly["trend"]    = trend_fn(monthly["month_num"])
    monthly["residual"] = monthly["sales"] / monthly["trend"]
    seasonal_factors    = monthly.groupby("month")["residual"].mean()

    # Du bao 6 thang tiep theo (2015-01 -> 2015-06)
    last_date    = monthly["order_date"].max()
    future_dates = pd.date_range(
        start=last_date + pd.DateOffset(months=1),
        periods=6, freq="MS"
    )
    future_month_nums = np.arange(len(monthly), len(monthly) + 6)
    future_months     = future_dates.month

    yhat        = trend_fn(future_month_nums)
    yhat_seas   = np.array([yhat[i] * seasonal_factors.get(m, 1.0)
                             for i, m in enumerate(future_months)])
    yhat_lower  = yhat_seas * 0.85
    yhat_upper  = yhat_seas * 1.15

    forecast_future = pd.DataFrame({
        "ds":          future_dates,
        "yhat":        yhat_seas,
        "yhat_lower":  yhat_lower,
        "yhat_upper":  yhat_upper,
    })

    # Fitted values cho lich su
    fitted_vals = np.array([
        trend_fn(row.month_num) * seasonal_factors.get(row.month, 1.0)
        for _, row in monthly.iterrows()
    ])
    forecast_hist = pd.DataFrame({
        "ds":   monthly["order_date"],
        "yhat": fitted_vals,
    })

    print(f"\nForecast next 6 months:")
    print(forecast_future[["ds","yhat","yhat_lower","yhat_upper"]].to_string(index=False))

# ── CHART 19: Forecast Overview ───────────────────────────────────────────────
print("\n[Chart 19] Sales Forecast Overview...")

fig, ax = plt.subplots(figsize=(13, 6))

# Actual data
ax.fill_between(monthly["order_date"], monthly["sales"]/1e3,
                alpha=0.12, color=BLUE)
ax.plot(monthly["order_date"], monthly["sales"]/1e3,
        color=BLUE, linewidth=2, label="Actual Sales", marker="o",
        markersize=4)

# Fitted / historical forecast
ax.plot(forecast_hist["ds"], forecast_hist["yhat"]/1e3,
        color=GRAY, linewidth=1.5, linestyle="--",
        alpha=0.8, label="Model Fit")

# Future forecast
ax.fill_between(forecast_future["ds"],
                forecast_future["yhat_lower"]/1e3,
                forecast_future["yhat_upper"]/1e3,
                alpha=0.2, color=GREEN, label="Confidence Interval")
ax.plot(forecast_future["ds"], forecast_future["yhat"]/1e3,
        color=GREEN, linewidth=2.5, linestyle="-",
        marker="s", markersize=6, label="Forecast (2015)")

# Them nhan cho forecast points
for _, row in forecast_future.iterrows():
    ax.annotate(f"${row['yhat']/1e3:.0f}K",
                xy=(row["ds"], row["yhat"]/1e3),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=8, color=GREEN,
                fontweight="bold")

# Duong phan cach actual vs forecast
split_date = monthly["order_date"].max()
ax.axvline(split_date, color="red", linewidth=1.5,
           linestyle=":", alpha=0.7, label="Forecast Start")
ax.text(split_date, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 500,
        "  Forecast →", color="red", fontsize=9)

# Q4 highlights
for year in [2011, 2012, 2013, 2014]:
    q4_start = pd.Timestamp(f"{year}-10-01")
    q4_end   = pd.Timestamp(f"{year}-12-31")
    ax.axvspan(q4_start, q4_end, alpha=0.06, color=RED)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate(rotation=30)

ax.set_ylabel("Monthly Sales (K USD)")
ax.set_title("Chart 19 — Sales Forecast: Actual vs Predicted (2011–2015)\n"
             "(Red shaded = Q4 peaks | Green line = 6-month forecast)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_19_forecast.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_19_forecast.png")

# ── CHART 20: YoY Comparison + Seasonality ────────────────────────────────────
print("[Chart 20] Seasonality Pattern...")

monthly["month"]     = monthly["order_date"].dt.month
monthly["year"]      = monthly["order_date"].dt.year
monthly["month_name"]= monthly["order_date"].dt.strftime("%b")

pivot = monthly.pivot_table(
    index="month", columns="year",
    values="sales", aggfunc="sum"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Line chart moi nam 1 duong
year_colors = {2011: "#BDC3C7", 2012: "#85C1E9",
               2013: "#2E75B6", 2014: "#1F4E79"}
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

for year in sorted(pivot.columns):
    data = pivot[year].values
    ax1.plot(range(1, len(data)+1), data/1e3,
             marker="o", markersize=4, linewidth=2,
             color=year_colors.get(year, BLUE),
             label=str(year))

ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_labels)
ax1.set_ylabel("Monthly Sales (K USD)")
ax1.set_title("YoY Monthly Sales Comparison\n(Darker = More recent)",
              fontweight="bold")
ax1.legend(title="Year")
ax1.axvspan(9.5, 12.5, alpha=0.08, color=RED, label="Q4 Peak")

# Bar chart average by month (seasonality)
avg_by_month = monthly.groupby("month")["sales"].mean()
colors_bar   = [RED if m in [10, 11, 12] else BLUE for m in avg_by_month.index]
bars = ax2.bar(range(1, 13), avg_by_month.values/1e3,
               color=colors_bar, width=0.7, alpha=0.85)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_labels)
ax2.set_ylabel("Avg Monthly Sales (K USD)")
ax2.set_title("Average Sales by Month (Seasonality)\n(Red = Q4 peak months)",
              fontweight="bold")

for bar, val in zip(bars, avg_by_month.values):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             f"${val/1e3:.0f}K",
             ha="center", fontsize=8)

plt.suptitle("Chart 20 — Sales Seasonality & YoY Comparison",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_20_seasonality.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_20_seasonality.png")

# ── Export forecast CSV cho Power BI ──────────────────────────────────────────
print("\n[Export] forecast_result.csv cho Power BI...")

# Lich su + forecast
history_export = pd.DataFrame({
    "date":       monthly["order_date"],
    "type":       "Actual",
    "sales":      monthly["sales"].round(0),
    "sales_lower":monthly["sales"].round(0),
    "sales_upper":monthly["sales"].round(0),
})

forecast_export = pd.DataFrame({
    "date":       forecast_future["ds"],
    "type":       "Forecast",
    "sales":      forecast_future["yhat"].round(0),
    "sales_lower":forecast_future["yhat_lower"].round(0),
    "sales_upper":forecast_future["yhat_upper"].round(0),
})

combined = pd.concat([history_export, forecast_export], ignore_index=True)
csv_path = os.path.join(DATA_DIR, "forecast_result.csv")
combined.to_csv(csv_path, index=False)
print(f"   Saved: {csv_path}")
print(f"   Rows : {len(combined)} ({len(history_export)} actual + {len(forecast_export)} forecast)")

# ── KEY INSIGHTS ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY INSIGHTS - SALES FORECAST")
print("=" * 60)

avg_q4  = monthly[monthly["month"].isin([10,11,12])]["sales"].mean()
avg_rest= monthly[~monthly["month"].isin([10,11,12])]["sales"].mean()
growth  = (monthly.groupby("year")["sales"].sum().pct_change() * 100).dropna()
total_forecast = forecast_future["yhat"].sum()

print(f"""
Seasonality:
  Q4 avg monthly sales : ${avg_q4:,.0f}
  Other months avg     : ${avg_rest:,.0f}
  Q4 premium           : +{(avg_q4/avg_rest - 1)*100:.0f}% above average

YoY Growth:
{chr(10).join([f"  {int(yr)} -> {int(yr)+1}: {g:+.1f}%" for yr, g in growth.items()])}

Forecast Summary (Jan-Jun 2015):
{chr(10).join([f"  {row.ds.strftime('%b %Y')}: ${row.yhat:,.0f}" for _, row in forecast_future.iterrows()])}
  Total H1 2015 forecast: ${total_forecast:,.0f}

Business Insights:
  1. Stock up inventory before October each year (Q4 demand spike)
  2. Plan marketing campaigns for Sep-Oct to capture Q4 momentum
  3. Jan-Feb consistently lowest sales — good time for promotions
  4. Overall trend is upward — business is growing year over year
""")

print(f" Charts: {CHART_DIR}")
print(f" CSV   : {csv_path}")
print("\n" + "=" * 60)
print("HOAN THANH Notebook 05 - Sales Forecast!")
print("")
print("=" * 60)
print("TAT CA 5 NOTEBOOKS PYTHON DA HOAN THANH!")
print("")
print("Charts tao ra (20 bieu do):")
print("  01_EDA              -> chart_01 den chart_06")
print("  02_Discount         -> chart_07 den chart_10")
print("  03_RFM              -> chart_11 den chart_14 + rfm_segments.csv")
print("  04_Geo              -> chart_15 den chart_18 + geo_summary.csv")
print("  05_Forecast         -> chart_19, chart_20   + forecast_result.csv")
print("")
print("Buoc tiep theo: Phase 3 - Power BI Dashboard!")
print("  - Mo Power BI Desktop")
print("  - Import: Global_Superstore2.xlsx + rfm_segments.csv + forecast_result.csv")
print("  - Xay dung Star Schema")
print("  - Viet 8 DAX measures")
print("  - Tao 5 trang dashboard")
print("=" * 60)

GLOBAL SUPERSTORE - SALES FORECASTING

Data loaded: 51,290 rows
Date range : 2011-01-01 -> 2014-12-31

Monthly data points: 48
Avg monthly sales  : $263,385
Max monthly sales  : $555,279

Prophet found - using Prophet for forecasting


12:06:29 - cmdstanpy - INFO - Chain [1] start processing
12:06:30 - cmdstanpy - INFO - Chain [1] done processing



Forecast next 6 months:
        ds          yhat    yhat_lower    yhat_upper
2015-01-01 267589.061307 245193.473854 291276.316843
2015-02-01 212873.823826 190314.843178 237185.876917
2015-03-01 361404.168657 338104.341880 383705.472104
2015-04-01 310976.081069 287535.633872 333470.856209
2015-05-01 362954.283075 339376.686384 388064.701148
2015-06-01 519800.406448 496463.478097 544051.795683

[Chart 19] Sales Forecast Overview...
   Saved: chart_19_forecast.png
[Chart 20] Seasonality Pattern...
   Saved: chart_20_seasonality.png

[Export] forecast_result.csv cho Power BI...
   Saved: C:\DA\global-superstore-analysis\data\forecast_result.csv
   Rows : 54 (48 actual + 6 forecast)

KEY INSIGHTS - SALES FORECAST

Seasonality:
  Q4 avg monthly sales : $358,354
  Other months avg     : $231,729
  Q4 premium           : +55% above average

YoY Growth:
  2012 -> 2013: +18.5%
  2013 -> 2014: +27.2%
  2014 -> 2015: +26.3%

Forecast Summary (Jan-Jun 2015):
  Jan 2015: $267,589
  Feb 2015: $212,8